In [12]:
!pip install lightgbm catboost scikit-learn pandas numpy scipy -q


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [13]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import find_peaks
from scipy.fft import rfft, rfftfreq
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.ensemble import ExtraTreesClassifier
import lightgbm as lgb
import catboost as cb
import os
from collections import Counter

# ── Configuration ──────────────────────────────────────────────────────
DATA_DIR = '/Users/dayana/git repo/machine-learning-class/lab6/knu-2026-machine-learning-final-assignment'
SEED = 42
N_FOLDS = 5

# ════════════════════════════════════════════════════════════════════════
# p7 COMPONENT FLAGS — toggle each proposed method on/off
# ════════════════════════════════════════════════════════════════════════
USE_SEED_ENSEMBLE   = False   # ① average base models over multiple seeds
USE_DEVICE_INVARIANT = True  # ③ add device-agnostic (global-norm) feature view
USE_SOFT_STAGE1     = False  # ② soft Stage-1 integration (no hard threshold)
USE_TEMPORAL_BOUT   = True   # ④ bout-consistency post-processing

SEED_LIST = [42]    # ① seeds for ensembling (3 = sweet spot)

# LGBM 9-class params (num_leaves 255, from p6)
LGBM_PARAMS = {
    'objective': 'multiclass', 'num_class': 9, 'metric': 'multi_logloss',
    'learning_rate': 0.05, 'num_leaves': 255, 'max_depth': -1,
    'min_child_samples': 20, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
    'bagging_freq': 1, 'lambda_l1': 0.1, 'lambda_l2': 0.1,
    'verbose': -1, 'n_jobs': -1, 'seed': SEED,
}
CAT_PARAMS = {
    'iterations': 4000, 'learning_rate': 0.05, 'depth': 10, 'l2_leaf_reg': 3,
    'loss_function': 'MultiClass', 'eval_metric': 'TotalF1:average=Macro',
    'early_stopping_rounds': 200, 'random_seed': SEED, 'verbose': 0,
    'allow_writing_files': False,
}
CAT_BINARY_PARAMS = {
    'iterations': 2000, 'learning_rate': 0.05, 'depth': 8, 'l2_leaf_reg': 3,
    'loss_function': 'Logloss', 'eval_metric': 'AUC',
    'early_stopping_rounds': 150, 'random_seed': SEED, 'verbose': 0,
    'allow_writing_files': False,
}
LGBM_BINARY_PARAMS = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'learning_rate': 0.05, 'num_leaves': 127, 'max_depth': -1,
    'min_child_samples': 20, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
    'bagging_freq': 1, 'lambda_l1': 0.1, 'lambda_l2': 0.1,
    'verbose': -1, 'n_jobs': -1, 'seed': SEED,
}
ET_PARAMS = {
    'n_estimators': 500, 'max_features': 'sqrt', 'min_samples_leaf': 5,
    'n_jobs': -1, 'random_state': SEED, 'class_weight': 'balanced',
}

print("Config loaded.")
print(f"Flags → seed_ens={USE_SEED_ENSEMBLE} dev_inv={USE_DEVICE_INVARIANT} "
      f"soft_s1={USE_SOFT_STAGE1} bout={USE_TEMPORAL_BOUT}")

Config loaded.
Flags → seed_ens=False dev_inv=True soft_s1=False bout=True


In [14]:
print("Loading sensor files (this may take ~1-2 minutes)...")
train_accel = pd.read_csv(os.path.join(DATA_DIR, 'train-accel.csv'))
train_gyro  = pd.read_csv(os.path.join(DATA_DIR, 'train-gyro.csv'))
train_label = pd.read_csv(os.path.join(DATA_DIR, 'train-label.csv'))
test_accel  = pd.read_csv(os.path.join(DATA_DIR, 'test-accel.csv'))
test_gyro   = pd.read_csv(os.path.join(DATA_DIR, 'test-gyro.csv'))
test_label  = pd.read_csv(os.path.join(DATA_DIR, 'test-label.csv'))

for name, df in [('train_accel',train_accel),('train_gyro',train_gyro),
                 ('train_label',train_label),('test_accel',test_accel),
                 ('test_gyro',test_gyro),('test_label',test_label)]:
    print(f"{name:12s}: {df.shape}")

# Keep ORIGINAL raw axes before per-device normalization (for ③ device-invariant view)
for df in [train_accel, test_accel, train_gyro, test_gyro]:
    df['x_raw'] = df['x'].copy()
    df['y_raw'] = df['y'].copy()
    df['z_raw'] = df['z'].copy()

def normalize_by_device(df_train, df_test, axes=['x','y','z']):
    df_train = df_train.copy(); df_test = df_test.copy()
    global_stats = {ax:(df_train[ax].mean(), df_train[ax].std()+1e-8) for ax in axes}
    known = set(df_train['device'].unique())
    for device in known:
        trm = df_train['device']==device; tem = df_test['device']==device
        for ax in axes:
            mu = df_train.loc[trm,ax].mean(); sig = df_train.loc[trm,ax].std()+1e-8
            df_train.loc[trm,ax] = (df_train.loc[trm,ax]-mu)/sig
            df_test.loc[tem,ax]  = (df_test.loc[tem,ax]-mu)/sig
    unk = ~df_test['device'].isin(known)
    if unk.sum()>0:
        print(f"  Global fallback for {unk.sum()} unknown-device rows.")
        for ax in axes:
            mu,sig = global_stats[ax]
            df_test.loc[unk,ax] = (df_test.loc[unk,ax]-mu)/sig
    return df_train, df_test

# ③ Global (device-agnostic) normalization view — uses combined train stats only
def normalize_global(df_train, df_test, axes=['x_raw','y_raw','z_raw'], suffix='_g'):
    df_train = df_train.copy(); df_test = df_test.copy()
    for ax in axes:
        mu = df_train[ax].mean(); sig = df_train[ax].std()+1e-8
        df_train[ax.replace('_raw','')+suffix] = (df_train[ax]-mu)/sig
        df_test[ax.replace('_raw','')+suffix]  = (df_test[ax]-mu)/sig
    return df_train, df_test

print("Per-device normalizing accel...")
train_accel, test_accel = normalize_by_device(train_accel, test_accel)
print("Per-device normalizing gyro...")
train_gyro,  test_gyro  = normalize_by_device(train_gyro,  test_gyro)

if USE_DEVICE_INVARIANT:
    print("Adding global (device-agnostic) normalization view...")
    train_accel, test_accel = normalize_global(train_accel, test_accel)
    train_gyro,  test_gyro  = normalize_global(train_gyro,  test_gyro)

print("Device normalization done.")
print(f"  train devices: {train_accel['device'].unique()}")
print(f"  test  devices: {test_accel['device'].unique()}")

Loading sensor files (this may take ~1-2 minutes)...
train_accel : (2428374, 7)
train_gyro  : (2433673, 7)
train_label : (38015, 4)
test_accel  : (2528310, 7)
test_gyro   : (2541831, 7)
test_label  : (39473, 4)
Per-device normalizing accel...
  Global fallback for 92768 unknown-device rows.
Per-device normalizing gyro...
  Global fallback for 92737 unknown-device rows.
Adding global (device-agnostic) normalization view...
Device normalization done.
  train devices: <StringArray>
['samsung', 'Apple']
Length: 2, dtype: str
  test  devices: <StringArray>
['unknown', 'samsung', 'Apple']
Length: 3, dtype: str


In [15]:
def apply_direction_correction(df):
    df = df.copy()
    df['xb'] = df['x'].copy(); df['yb'] = df['y'].copy(); df['zb'] = df['z'].copy()
    for d,(sx,sy,sz) in {1:('y','-x','z'),2:('-y','x','z'),
                         3:('y','-x','-z'),4:('-y','x','-z')}.items():
        m = df['direction']==d
        df.loc[m,'xb'] = (df.loc[m,'x'] if sx=='x' else -df.loc[m,'x'] if sx=='-x'
                          else df.loc[m,'y'] if sx=='y' else -df.loc[m,'y'])
        df.loc[m,'yb'] = (df.loc[m,'x'] if sy=='x' else -df.loc[m,'x'] if sy=='-x'
                          else df.loc[m,'y'] if sy=='y' else -df.loc[m,'y'])
        df.loc[m,'zb'] = (df.loc[m,'z'] if sz=='z' else -df.loc[m,'z'])
    df['magb'] = np.sqrt(df['xb']**2 + df['yb']**2 + df['zb']**2)

    # ③ device-invariant body-frame magnitude (from global-normalized axes)
    if 'x_g' in df.columns:
        df['magb_g'] = np.sqrt(df['x_g']**2 + df['y_g']**2 + df['z_g']**2)
    return df

print("Direction correction: accel...")
train_accel = apply_direction_correction(train_accel)
test_accel  = apply_direction_correction(test_accel)
print("Direction correction: gyro...")
train_gyro  = apply_direction_correction(train_gyro)
test_gyro   = apply_direction_correction(test_gyro)
print("Direction correction done.")

Direction correction: accel...
Direction correction: gyro...
Direction correction done.


In [16]:
FS = 50.0

def stat_features(arr, prefix):
    feats = {}; n = len(arr)
    if n == 0:
        for k in ['mean','std','var','min','max','range','median','p25','p75','iqr',
                  'skew','kurt','rms','energy','zero_cross','mean_abs_diff','max_abs']:
            feats[f'{prefix}_{k}'] = 0.0
        return feats
    feats[f'{prefix}_mean']=np.mean(arr); feats[f'{prefix}_std']=np.std(arr)
    feats[f'{prefix}_var']=np.var(arr); feats[f'{prefix}_min']=np.min(arr)
    feats[f'{prefix}_max']=np.max(arr); feats[f'{prefix}_range']=np.ptp(arr)
    feats[f'{prefix}_median']=np.median(arr); feats[f'{prefix}_p25']=np.percentile(arr,25)
    feats[f'{prefix}_p75']=np.percentile(arr,75)
    feats[f'{prefix}_iqr']=np.percentile(arr,75)-np.percentile(arr,25)
    feats[f'{prefix}_skew']=float(stats.skew(arr)) if n>2 else 0.0
    feats[f'{prefix}_kurt']=float(stats.kurtosis(arr)) if n>2 else 0.0
    feats[f'{prefix}_rms']=np.sqrt(np.mean(arr**2)); feats[f'{prefix}_energy']=np.sum(arr**2)/n
    feats[f'{prefix}_zero_cross']=np.sum(np.diff(np.sign(arr-np.mean(arr)))!=0)
    feats[f'{prefix}_mean_abs_diff']=np.mean(np.abs(np.diff(arr))) if n>1 else 0.0
    feats[f'{prefix}_max_abs']=np.max(np.abs(arr))
    return feats

def fft_features(arr, prefix, fs=FS):
    feats = {}; n = len(arr)
    if n < 8:
        for k in ['dom_freq','dom_amp','spectral_entropy','band_low','band_mid',
                  'band_high','band_ratio_mid_low']:
            feats[f'{prefix}_{k}'] = 0.0
        return feats
    ac = arr-np.mean(arr); fv = np.abs(rfft(ac)); fr = rfftfreq(n,d=1.0/fs)
    tp = np.sum(fv**2)+1e-10; di = np.argmax(fv)
    feats[f'{prefix}_dom_freq']=fr[di]; feats[f'{prefix}_dom_amp']=fv[di]
    pn = fv**2/tp; pn = pn[pn>0]
    feats[f'{prefix}_spectral_entropy']=-np.sum(pn*np.log(pn))
    lm=(fr>=0.0)&(fr<1.0); mm=(fr>=1.0)&(fr<3.0); hm=(fr>=3.0)&(fr<8.0)
    feats[f'{prefix}_band_low']=np.sum(fv[lm]**2)/tp
    feats[f'{prefix}_band_mid']=np.sum(fv[mm]**2)/tp
    feats[f'{prefix}_band_high']=np.sum(fv[hm]**2)/tp
    feats[f'{prefix}_band_ratio_mid_low']=feats[f'{prefix}_band_mid']/(feats[f'{prefix}_band_low']+1e-10)
    return feats

def peak_features(arr, prefix, fs=FS):
    feats = {}; n = len(arr)
    if n < 4:
        for k in ['n_peaks','peak_rate','mean_peak_height','std_peak_height',
                  'mean_peak_interval','peak_regularity']:
            feats[f'{prefix}_{k}'] = 0.0
        return feats
    ac = arr-np.mean(arr); ht = 0.3*np.std(ac)
    peaks,props = find_peaks(ac, height=ht, distance=int(fs*0.15))
    npk = len(peaks); dur = n/fs
    feats[f'{prefix}_n_peaks']=npk; feats[f'{prefix}_peak_rate']=npk/dur
    if npk>0:
        ph = props['peak_heights']
        feats[f'{prefix}_mean_peak_height']=np.mean(ph)
        feats[f'{prefix}_std_peak_height']=np.std(ph)
    else:
        feats[f'{prefix}_mean_peak_height']=0.0; feats[f'{prefix}_std_peak_height']=0.0
    if npk>1:
        iv = np.diff(peaks)/fs
        feats[f'{prefix}_mean_peak_interval']=np.mean(iv)
        feats[f'{prefix}_peak_regularity']=1.0/(np.std(iv)+1e-8)
    else:
        feats[f'{prefix}_mean_peak_interval']=0.0; feats[f'{prefix}_peak_regularity']=0.0
    return feats

def posture_features(a_xb, a_yb, a_zb):
    feats = {}
    if len(a_xb)==0:
        feats.update({'posture_zb_mean':0.0,'posture_yb_mean':0.0,'posture_tilt_angle':0.0,
                      'posture_prone_score':0.0,'posture_vertical_energy':0.0})
        return feats
    gx=np.mean(a_xb); gy=np.mean(a_yb); gz=np.mean(a_zb)
    gm=np.sqrt(gx**2+gy**2+gz**2)+1e-8
    feats['posture_zb_mean']=gz; feats['posture_yb_mean']=gy
    feats['posture_tilt_angle']=np.arccos(np.clip(abs(gy)/gm,-1,1))
    feats['posture_prone_score']=abs(gz)/gm
    dz=np.array(a_zb)-gz; feats['posture_vertical_energy']=np.mean(dz**2)
    return feats

def jerk_features(arr, prefix, fs=FS):
    feats = {}
    keys=['mean_abs','std','max_abs','rms','energy','p90_abs','p95_abs','skew','kurt',
          'n_peaks','peak_rate','mean_peak_height','impulse_ratio']
    if len(arr)<2:
        for k in keys: feats[f'{prefix}_{k}']=0.0
        return feats
    jerk=np.diff(arr.astype(np.float64))*fs; n=len(jerk)
    feats[f'{prefix}_mean_abs']=np.mean(np.abs(jerk)); feats[f'{prefix}_std']=np.std(jerk)
    feats[f'{prefix}_max_abs']=np.max(np.abs(jerk)); feats[f'{prefix}_rms']=np.sqrt(np.mean(jerk**2))
    feats[f'{prefix}_energy']=np.sum(jerk**2)/n
    feats[f'{prefix}_p90_abs']=np.percentile(np.abs(jerk),90)
    feats[f'{prefix}_p95_abs']=np.percentile(np.abs(jerk),95)
    feats[f'{prefix}_skew']=float(stats.skew(jerk)) if n>2 else 0.0
    feats[f'{prefix}_kurt']=float(stats.kurtosis(jerk)) if n>2 else 0.0
    aj=np.abs(jerk); th=0.5*np.std(aj)
    peaks,props=find_peaks(aj,height=th,distance=int(fs*0.1))
    npk=len(peaks); dur=n/fs
    feats[f'{prefix}_n_peaks']=npk; feats[f'{prefix}_peak_rate']=npk/dur
    feats[f'{prefix}_mean_peak_height']=np.mean(props['peak_heights']) if npk>0 else 0.0
    feats[f'{prefix}_impulse_ratio']=feats[f'{prefix}_max_abs']/(feats[f'{prefix}_mean_abs']+1e-10)
    return feats

def lateral_asymmetry_features(axb, prefix='lat'):
    feats = {}
    keys=['xb_skew','xb_kurt','xb_asymmetry_idx','xb_pos_ratio','xb_range_ratio','xb_mean_abs']
    arr=np.asarray(axb,dtype=np.float64); arr=arr[np.isfinite(arr)]
    if len(arr)<4:
        for k in keys: feats[f'{prefix}_{k}']=0.0
        return feats
    ac=arr-np.mean(arr)
    feats[f'{prefix}_xb_skew']=float(stats.skew(ac)) if len(ac)>2 else 0.0
    feats[f'{prefix}_xb_kurt']=float(stats.kurtosis(ac)) if len(ac)>2 else 0.0
    feats[f'{prefix}_xb_asymmetry_idx']=float(abs(np.mean(arr))/(np.std(arr)+1e-8))
    pv=arr[arr>0]; feats[f'{prefix}_xb_pos_ratio']=float(len(pv)/len(arr))
    feats[f'{prefix}_xb_range_ratio']=float(np.max(ac)/(abs(np.min(ac))+1e-8))
    feats[f'{prefix}_xb_mean_abs']=float(np.mean(np.abs(ac)))
    return feats

def autocorr_features(arr, prefix, fs=FS, lags_s=(0.5,1.0,1.5,2.0,2.5)):
    feats={}; ln=[str(l).replace('.','p') for l in lags_s]
    keys=[f'lag_{n}' for n in ln]+['max','mean','std','best_lag_s']
    arr=np.asarray(arr,dtype=np.float64); arr=arr[np.isfinite(arr)]
    if len(arr)<int(max(lags_s)*fs)+2 or np.std(arr)<1e-8:
        for k in keys: feats[f'{prefix}_{k}']=0.0
        return feats
    x=arr-np.mean(arr); x=x/(np.std(x)+1e-8); vals=[]
    for ls,nm in zip(lags_s,ln):
        lag=int(round(ls*fs))
        ac=0.0 if (lag<=0 or len(x)<=lag) else float(np.mean(x[:-lag]*x[lag:]))
        feats[f'{prefix}_lag_{nm}']=ac; vals.append(ac)
    vals=np.asarray(vals,dtype=np.float64)
    feats[f'{prefix}_max']=float(np.max(vals)); feats[f'{prefix}_mean']=float(np.mean(vals))
    feats[f'{prefix}_std']=float(np.std(vals))
    feats[f'{prefix}_best_lag_s']=float(lags_s[int(np.argmax(vals))])
    return feats

print("Feature helpers defined.")

Feature helpers defined.


In [17]:
CONTEXT_SHORT   = 1
CONTEXT_LONG    = 4
CONTEXT_FFT_LONG = 6

def compute_features_p7(accel_df, gyro_df, label_df):
    accel_df = accel_df.copy()
    gyro_df  = gyro_df.copy()
    label_df = label_df.copy()

    # CRITICAL: reset index so grp[cols].values aligns correctly
    accel_df = accel_df.reset_index(drop=True)
    gyro_df  = gyro_df.reset_index(drop=True)

    accel_df['time_s'] = accel_df['time'].astype(int)
    gyro_df['time_s']  = gyro_df['time'].astype(int)
    label_df['time_s'] = label_df['time'].astype(int)

    for df in [accel_df, gyro_df]:
        df['mag'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

    print("  Building sensor lookup tables...")
    def build_lookup(df, cols):
        lookup = {}
        for (pid, ts), grp in df.groupby(['pid', 'time_s']):
            lookup[(pid, ts)] = grp[cols].values.astype(np.float32)
        return lookup

    accel_raw_lookup  = build_lookup(accel_df, ['x','y','z','mag'])
    accel_body_lookup = build_lookup(accel_df, ['xb','yb','zb','magb'])
    gyro_raw_lookup   = build_lookup(gyro_df,  ['x','y','z','mag'])
    gyro_body_lookup  = build_lookup(gyro_df,  ['xb','yb','zb','magb'])

    meta_lookup = {}
    for (pid, ts), grp in accel_df.groupby(['pid', 'time_s']):
        meta_lookup[(pid, ts)] = {
            'direction': grp['direction'].iloc[0],
            'device':    1 if str(grp['device'].iloc[0]).lower() == 'apple' else 0,
        }

    print("  Extracting features per label row...")
    all_feats = []
    for idx, (_, row) in enumerate(label_df.iterrows()):
        pid = row['pid']; ts = int(row['time_s']); feat = {}

        # SHORT window (±1s)
        ax_s, ay_s, az_s, amag_s = [], [], [], []
        gx_s, gy_s, gz_s, gmag_s = [], [], [], []
        axb_s, ayb_s, azb_s      = [], [], []
        for offset in range(-CONTEXT_SHORT, CONTEXT_SHORT + 1):
            key = (pid, ts + offset)
            if key in accel_raw_lookup:
                c = accel_raw_lookup[key]
                ax_s.extend(c[:,0]); ay_s.extend(c[:,1]); az_s.extend(c[:,2]); amag_s.extend(c[:,3])
            if key in accel_body_lookup:
                c = accel_body_lookup[key]
                axb_s.extend(c[:,0]); ayb_s.extend(c[:,1]); azb_s.extend(c[:,2])
            if key in gyro_raw_lookup:
                c = gyro_raw_lookup[key]
                gx_s.extend(c[:,0]); gy_s.extend(c[:,1]); gz_s.extend(c[:,2]); gmag_s.extend(c[:,3])

        # LONG window (±4s)
        amag_l, axb_l, ayb_l, azb_l = [], [], [], []
        gmag_l, gxb_l, gyb_l, gzb_l = [], [], [], []
        for offset in range(-CONTEXT_LONG, CONTEXT_LONG + 1):
            key = (pid, ts + offset)
            if key in accel_raw_lookup:
                c = accel_raw_lookup[key]; amag_l.extend(c[:,3])
            if key in accel_body_lookup:
                c = accel_body_lookup[key]
                axb_l.extend(c[:,0]); ayb_l.extend(c[:,1]); azb_l.extend(c[:,2])
            if key in gyro_raw_lookup:
                c = gyro_raw_lookup[key]; gmag_l.extend(c[:,3])
            if key in gyro_body_lookup:
                c = gyro_body_lookup[key]
                gxb_l.extend(c[:,0]); gyb_l.extend(c[:,1]); gzb_l.extend(c[:,2])

        # EXTENDED LONG window (±6s)
        amag_xl, ayb_xl, azb_xl = [], [], []
        gmag_xl, gyb_xl         = [], []
        for offset in range(-CONTEXT_FFT_LONG, CONTEXT_FFT_LONG + 1):
            key = (pid, ts + offset)
            if key in accel_raw_lookup:
                c = accel_raw_lookup[key]; amag_xl.extend(c[:,3])
            if key in accel_body_lookup:
                c = accel_body_lookup[key]
                ayb_xl.extend(c[:,1]); azb_xl.extend(c[:,2])
            if key in gyro_raw_lookup:
                c = gyro_raw_lookup[key]; gmag_xl.extend(c[:,3])
            if key in gyro_body_lookup:
                c = gyro_body_lookup[key]; gyb_xl.extend(c[:,1])

        # numpy conversions
        ax_s   = np.array(ax_s,   dtype=np.float32)
        ay_s   = np.array(ay_s,   dtype=np.float32)
        az_s   = np.array(az_s,   dtype=np.float32)
        amag_s = np.array(amag_s, dtype=np.float32)
        gx_s   = np.array(gx_s,   dtype=np.float32)
        gy_s   = np.array(gy_s,   dtype=np.float32)
        gz_s   = np.array(gz_s,   dtype=np.float32)
        gmag_s = np.array(gmag_s, dtype=np.float32)
        axb_s  = np.array(axb_s,  dtype=np.float32)
        ayb_s  = np.array(ayb_s,  dtype=np.float32)
        azb_s  = np.array(azb_s,  dtype=np.float32)

        amag_l = np.array(amag_l, dtype=np.float32)
        axb_l  = np.array(axb_l,  dtype=np.float32)
        ayb_l  = np.array(ayb_l,  dtype=np.float32)
        azb_l  = np.array(azb_l,  dtype=np.float32)
        gmag_l = np.array(gmag_l, dtype=np.float32)
        gxb_l  = np.array(gxb_l,  dtype=np.float32)
        gyb_l  = np.array(gyb_l,  dtype=np.float32)
        gzb_l  = np.array(gzb_l,  dtype=np.float32)

        amag_xl = np.array(amag_xl, dtype=np.float32)
        ayb_xl  = np.array(ayb_xl,  dtype=np.float32)
        azb_xl  = np.array(azb_xl,  dtype=np.float32)
        gmag_xl = np.array(gmag_xl, dtype=np.float32)
        gyb_xl  = np.array(gyb_xl,  dtype=np.float32)

        # STAT FEATURES
        feat.update(stat_features(ax_s,   'ax'))
        feat.update(stat_features(ay_s,   'ay'))
        feat.update(stat_features(az_s,   'az'))
        feat.update(stat_features(amag_s, 'amag'))
        feat.update(stat_features(axb_s,  'axb'))
        feat.update(stat_features(ayb_s,  'ayb'))
        feat.update(stat_features(azb_s,  'azb'))
        feat.update(stat_features(gx_s,   'gx'))
        feat.update(stat_features(gy_s,   'gy'))
        feat.update(stat_features(gz_s,   'gz'))
        feat.update(stat_features(gmag_s, 'gmag'))

        if len(amag_s) > 2 and len(gmag_s) > 2:
            ml = min(len(amag_s), len(gmag_s))
            feat['accel_gyro_mag_corr'] = float(np.corrcoef(amag_s[:ml], gmag_s[:ml])[0, 1])
        else:
            feat['accel_gyro_mag_corr'] = 0.0

        # FFT FEATURES
        feat.update(fft_features(amag_l, 'amag_fft'))
        feat.update(fft_features(axb_l,  'axb_fft'))
        feat.update(fft_features(ayb_l,  'ayb_fft'))
        feat.update(fft_features(azb_l,  'azb_fft'))
        feat.update(fft_features(gmag_l, 'gmag_fft'))
        feat.update(fft_features(gxb_l,  'gxb_fft'))
        feat.update(fft_features(gyb_l,  'gyb_fft'))
        feat.update(fft_features(gzb_l,  'gzb_fft'))

        # EXTENDED FFT
        feat.update(fft_features(amag_xl, 'amag_xfft'))
        feat.update(fft_features(ayb_xl,  'ayb_xfft'))
        feat.update(fft_features(azb_xl,  'azb_xfft'))
        feat.update(fft_features(gmag_xl, 'gmag_xfft'))
        feat.update(fft_features(gyb_xl,  'gyb_xfft'))

        # PEAK FEATURES
        feat.update(peak_features(amag_l, 'amag_pk'))
        feat.update(peak_features(axb_l,  'axb_pk'))
        feat.update(peak_features(ayb_l,  'ayb_pk'))
        feat.update(peak_features(azb_l,  'azb_pk'))
        feat.update(peak_features(gmag_l, 'gmag_pk'))

        # POSTURE
        feat.update(posture_features(axb_s, ayb_s, azb_s))

        # JERK FEATURES
        feat.update(jerk_features(amag_s, 'amag_jerk_s'))
        feat.update(jerk_features(azb_s,  'azb_jerk_s'))
        feat.update(jerk_features(gmag_s, 'gmag_jerk_s'))
        feat.update(jerk_features(amag_l, 'amag_jerk_l'))
        feat.update(jerk_features(azb_l,  'azb_jerk_l'))
        feat.update(jerk_features(gmag_l, 'gmag_jerk_l'))

        # AUTOCORRELATION
        feat.update(autocorr_features(amag_l, 'amag_acorr_l'))
        feat.update(autocorr_features(azb_l,  'azb_acorr_l'))
        feat.update(autocorr_features(gmag_l, 'gmag_acorr_l'))
        feat.update(autocorr_features(ayb_l,  'ayb_acorr_l'))
        feat.update(autocorr_features(axb_l,  'axb_acorr_l'))
        feat.update(autocorr_features(gxb_l,  'gxb_acorr_l'))
        feat.update(autocorr_features(gyb_l,  'gyb_acorr_l'))

        # LATERAL ASYMMETRY
        feat.update(lateral_asymmetry_features(axb_s, prefix='lat_s'))
        feat.update(lateral_asymmetry_features(axb_l, prefix='lat_l'))

        # GRAVITY + TILT
        if len(ax_s) > 0:
            feat['gravity_x']   = float(np.mean(ax_s))
            feat['gravity_y']   = float(np.mean(ay_s))
            feat['gravity_z']   = float(np.mean(az_s))
            gm = np.sqrt(feat['gravity_x']**2 + feat['gravity_y']**2 + feat['gravity_z']**2)
            feat['gravity_mag'] = gm
            feat['tilt_xz']     = float(np.arctan2(feat['gravity_x'], feat['gravity_z'] + 1e-8))
            feat['tilt_yz']     = float(np.arctan2(feat['gravity_y'], feat['gravity_z'] + 1e-8))
        else:
            feat.update({'gravity_x':0,'gravity_y':0,'gravity_z':0,
                         'gravity_mag':0,'tilt_xz':0,'tilt_yz':0})

        # Data quality
        feat['n_accel_samples'] = len(ax_s)
        feat['n_gyro_samples']  = len(gx_s)

        # Metadata
        if (pid, ts) in meta_lookup:
            feat['direction'] = meta_lookup
            feat['device']    = meta_lookup
        else:
            feat['direction'] = -1
            feat['device']    = -1

        all_feats.append(feat)
        if (idx + 1) % 5000 == 0:
            print(f"    {idx+1}/{len(label_df)} rows done...")

    feat_df = pd.DataFrame(all_feats)
    feat_df.index = label_df.index
    return feat_df

print("p7 feature extraction defined (p6 method, no device-invariant).")

p7 feature extraction defined (p6 method, no device-invariant).


In [18]:
print("Extracting TRAIN features...")
train_feats = compute_features_p7(train_accel, train_gyro, train_label)
print(f"Train features: {train_feats.shape}")

print("\nExtracting TEST features...")
test_feats = compute_features_p7(test_accel, test_gyro, test_label)
print(f"Test features: {test_feats.shape}")

Extracting TRAIN features...
  Building sensor lookup tables...
  Extracting features per label row...
    5000/38015 rows done...
    10000/38015 rows done...
    15000/38015 rows done...
    20000/38015 rows done...
    25000/38015 rows done...
    30000/38015 rows done...
    35000/38015 rows done...
Train features: (38015, 477)

Extracting TEST features...
  Building sensor lookup tables...
  Extracting features per label row...
    5000/39473 rows done...
    10000/39473 rows done...
    15000/39473 rows done...
    20000/39473 rows done...
    25000/39473 rows done...
    30000/39473 rows done...
    35000/39473 rows done...
Test features: (39473, 477)


In [19]:
def add_interaction_features(df):
    df=df.copy(); eps=1e-10
    df['inter_gyb_dom_amp_over_gy_iqr']=df['gyb_fft_dom_amp']/(df['gy_iqr']+eps)
    df['inter_ayb_band_low_over_mid']=df['ayb_fft_band_low']/(df['ayb_fft_band_mid']+eps)
    df['inter_tilt_x_amag_std']=df['posture_tilt_angle']*df['amag_std']
    df['inter_gxb_mid_over_gmag_mid']=df['gxb_fft_band_mid']/(df['gmag_fft_band_mid']+eps)
    df['inter_azb_acorr_x_ayb_dom_amp']=df['azb_acorr_l_best_lag_s']*df['ayb_fft_dom_amp']
    df['inter_gyb_entropy_over_amag_entropy']=df['gyb_fft_spectral_entropy']/(df['amag_fft_spectral_entropy']+eps)
    df['inter_ayb_acorr_over_azb_acorr']=df['ayb_acorr_l_max']/(df['azb_acorr_l_max']+eps)
    df['inter_ayb_xfft_low_over_fft_low']=df['ayb_xfft_band_low']/(df['ayb_fft_band_low']+eps)
    return df

train_feats=add_interaction_features(train_feats)
test_feats =add_interaction_features(test_feats)

# Force every column to numeric; dict/object cells -> NaN -> 0
train_feats_model = train_feats.apply(pd.to_numeric, errors='coerce').fillna(0).replace([np.inf,-np.inf],0)
test_feats_model  = test_feats.apply(pd.to_numeric, errors='coerce').fillna(0).replace([np.inf,-np.inf],0)

feat_cols=list(train_feats_model.columns)
test_feats_model=test_feats_model.reindex(columns=feat_cols,fill_value=0)

X_train=train_feats_model.values.astype(np.float32)
y_train=train_label['workout'].values
groups =train_label['pid'].values
train_times=train_label['time'].astype(int).values
X_test=test_feats_model.values.astype(np.float32)
test_times=test_label['time'].astype(int).values

pid71_mask=(test_label['pid']=='71').values
has_sensor_mask=~pid71_mask

print(f"Feature cols: {len(feat_cols)} | X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"PID-71 rows: {pid71_mask.sum()} | with sensor: {has_sensor_mask.sum()}")

class_counts=Counter(y_train); n_total=len(y_train); n_classes=9
class_weights={c:n_total/(n_classes*class_counts[c]) for c in range(n_classes)}
sample_weights_train=np.array([class_weights[y] for y in y_train],dtype=np.float32)
print("Class counts:", dict(sorted(class_counts.items())))

Feature cols: 485 | X_train: (38015, 485) | X_test: (39473, 485)
PID-71 rows: 1124 | with sensor: 38349
Class counts: {np.int64(0): 3389, np.int64(1): 3523, np.int64(2): 3575, np.int64(3): 3697, np.int64(4): 3610, np.int64(5): 3627, np.int64(6): 3589, np.int64(7): 3795, np.int64(8): 9210}


In [20]:
y_binary=(y_train==8).astype(int)
rest_ratio=class_counts[8]/n_total; active_ratio=1.0-rest_ratio
binary_weights_map={0:1.0/active_ratio, 1:0.8/rest_ratio}
sample_weights_binary=np.array([binary_weights_map[y] for y in y_binary],dtype=np.float32)
sample_weights_binary/=sample_weights_binary.mean()

gkf=GroupKFold(n_splits=N_FOLDS)
print("="*60); print("STAGE 1: Binary Active/Rest"); print("="*60)

oof_binary_cat=np.zeros(len(X_train)); oof_binary_lgbm=np.zeros(len(X_train))
pred_binary_cat=np.zeros(len(X_test)); pred_binary_lgbm=np.zeros(len(X_test))

for fold,(tr,val) in enumerate(gkf.split(X_train,y_train,groups)):
    Xtr,Xval=X_train[tr],X_train[val]; ytr,yval=y_binary[tr],y_binary[val]; sw=sample_weights_binary[tr]
    cm=cb.CatBoostClassifier(**CAT_BINARY_PARAMS)
    cm.fit(Xtr,ytr,sample_weight=sw,eval_set=(Xval,yval),use_best_model=True,verbose=False)
    oof_binary_cat[val]=cm.predict_proba(Xval)[:,1]
    pred_binary_cat+=cm.predict_proba(X_test)[:,1]/N_FOLDS
    dtr=lgb.Dataset(Xtr,label=ytr,weight=sw,feature_name=feat_cols)
    dvl=lgb.Dataset(Xval,label=yval,feature_name=feat_cols,reference=dtr)
    lm=lgb.train(LGBM_BINARY_PARAMS,dtr,num_boost_round=2000,valid_sets=[dvl],
                 callbacks=[lgb.early_stopping(100,verbose=False),lgb.log_evaluation(-1)])
    oof_binary_lgbm[val]=lm.predict(Xval,num_iteration=lm.best_iteration)
    pred_binary_lgbm+=lm.predict(X_test,num_iteration=lm.best_iteration)/N_FOLDS
    vr=((0.5*oof_binary_cat[val]+0.5*oof_binary_lgbm[val])>0.5).astype(int)
    rr=np.mean(vr[yval==1]==1) if (yval==1).sum()>0 else 0
    ar=np.mean(vr[yval==0]==0) if (yval==0).sum()>0 else 0
    print(f"  Fold {fold+1} | Rest rec={rr:.4f} | Active rec={ar:.4f}")

oof_binary_blend=0.5*oof_binary_cat+0.5*oof_binary_lgbm
pred_binary_blend=0.5*pred_binary_cat+0.5*pred_binary_lgbm

# threshold tuning (still computed; used only if USE_SOFT_STAGE1=False)
best_thresh=0.5; best_proxy=-1
for th in np.arange(0.40,0.76,0.02):
    pr=(oof_binary_blend>th).astype(int)
    ar=np.mean(pr[y_binary==0]==0); rr=np.mean(pr[y_binary==1]==1)
    proxy=(8*ar+rr)/9
    if proxy>best_proxy: best_proxy=proxy; best_thresh=th
print(f"  Best binary threshold (hard mode): {best_thresh:.2f} (proxy={best_proxy:.4f})")
oof_is_active=(oof_binary_blend<=best_thresh)
test_is_active=(pred_binary_blend<=best_thresh)

STAGE 1: Binary Active/Rest
  Fold 1 | Rest rec=0.9735 | Active rec=0.9407
  Fold 2 | Rest rec=0.9513 | Active rec=0.9909
  Fold 3 | Rest rec=0.9039 | Active rec=0.9616
  Fold 4 | Rest rec=0.9800 | Active rec=0.9786
  Fold 5 | Rest rec=0.9537 | Active rec=0.9476
  Best binary threshold (hard mode): 0.74 (proxy=0.9671)


In [21]:
print("="*60); print("STAGE 2: Seed-ensembled base models (LGBM/Cat/ET)"); print("="*60)
seeds = SEED_LIST if USE_SEED_ENSEMBLE else [SEED]
print(f"Seeds: {seeds}")

oof_lgbm=np.zeros((len(X_train),9)); pred_lgbm=np.zeros((len(X_test),9))
oof_cat =np.zeros((len(X_train),9)); pred_cat =np.zeros((len(X_test),9))
oof_et  =np.zeros((len(X_train),9)); pred_et  =np.zeros((len(X_test),9))

for fold,(tr,val) in enumerate(gkf.split(X_train,y_train,groups)):
    Xtr,Xval=X_train[tr],X_train[val]; ytr,yval=y_train[tr],y_train[val]; sw=sample_weights_train[tr]
    lgb_v=np.zeros((len(val),9)); lgb_t=np.zeros((len(X_test),9))
    cat_v=np.zeros((len(val),9)); cat_t=np.zeros((len(X_test),9))
    et_v =np.zeros((len(val),9)); et_t =np.zeros((len(X_test),9))
    for s in seeds:
        p=dict(LGBM_PARAMS); p['seed']=s
        dtr=lgb.Dataset(Xtr,label=ytr,weight=sw,feature_name=feat_cols)
        dvl=lgb.Dataset(Xval,label=yval,feature_name=feat_cols,reference=dtr)
        m=lgb.train(p,dtr,num_boost_round=3000,valid_sets=[dvl],
                    callbacks=[lgb.early_stopping(100,verbose=False),lgb.log_evaluation(-1)])
        lgb_v+=m.predict(Xval,num_iteration=m.best_iteration)/len(seeds)
        lgb_t+=m.predict(X_test,num_iteration=m.best_iteration)/len(seeds)

        cp=dict(CAT_PARAMS); cp['random_seed']=s
        cmodel=cb.CatBoostClassifier(**cp)
        cmodel.fit(Xtr,ytr,sample_weight=sw,eval_set=(Xval,yval),use_best_model=True,verbose=False)
        cat_v+=cmodel.predict_proba(Xval)/len(seeds)
        cat_t+=cmodel.predict_proba(X_test)/len(seeds)

    oof_lgbm[val]=lgb_v; pred_lgbm+=lgb_t/N_FOLDS
    oof_cat[val]=cat_v;  pred_cat +=cat_t/N_FOLDS
    oof_et[val]=et_v;    pred_et  +=et_t/N_FOLDS

    s_l=balanced_accuracy_score(yval,np.argmax(lgb_v,1))
    s_c=balanced_accuracy_score(yval,np.argmax(cat_v,1))
    s_e=balanced_accuracy_score(yval,np.argmax(et_v,1))
    print(f"  Fold {fold+1} | LGBM {s_l:.5f} | Cat {s_c:.5f} | ET {s_e:.5f}")

print(f"\nLGBM OOF BA: {balanced_accuracy_score(y_train,np.argmax(oof_lgbm,1)):.5f}")
print(f"Cat  OOF BA: {balanced_accuracy_score(y_train,np.argmax(oof_cat,1)):.5f}")
print(f"ET   OOF BA: {balanced_accuracy_score(y_train,np.argmax(oof_et,1)):.5f}")

STAGE 2: Seed-ensembled base models (LGBM/Cat/ET)
Seeds: [42]
  Fold 1 | LGBM 0.88346 | Cat 0.89636 | ET 0.11111


KeyboardInterrupt: 

In [ ]:
print("Searching 3-model blend weights...")
rows=[]
for w_l in np.round(np.arange(0.0, 1.01, 0.05), 2):
    w_c = round(1.0 - w_l, 2)
    b = w_l*oof_lgbm + w_c*oof_cat
    rows.append({'w_lgbm': w_l, 'w_cat': w_c, 'raw_oof_ba': balanced_accuracy_score(y_train, np.argmax(b,1))})

blend_df=pd.DataFrame(rows).sort_values('raw_oof_ba',ascending=False)
print("\nTop 10 raw blends:")
print(blend_df.head(10).to_string(index=False))
best_raw=blend_df.iloc[0]
print(f"\nBest raw: LGBM={best_raw['w_lgbm']:.2f} Cat={best_raw['w_cat']:.2f} "
      f"ET={best_raw['w_et']:.2f} | BA={best_raw['raw_oof_ba']:.5f}")

In [ ]:
# ── Soft probability smoothing (from p6) ───────────────────────────────
def soft_prob_smooth(prob_matrix, pids, times, window=5):
    prob_matrix=np.asarray(prob_matrix); pids=np.asarray(pids); times=np.asarray(times)
    smoothed=prob_matrix.copy(); hw=window//2
    for pid in np.unique(pids):
        idx=np.where(pids==pid)[0]
        if len(idx)<window: continue
        oidx=idx[np.argsort(times[idx],kind='mergesort')]
        lp=prob_matrix[oidx]; ls=np.zeros_like(lp)
        for i in range(len(lp)):
            lo=max(0,i-hw); hi=min(len(lp),i+hw+1)
            ls[i]=lp[lo:hi].mean(axis=0)
        smoothed[oidx]=ls
    return smoothed

# ── ② SOFT Stage-1 integration (no hard threshold) ─────────────────────
# Multiply 9-class Rest prob by Stage-1 Rest prob; down-weight Rest in
# active rows softly via (1 - p_rest_stage1). No cliff, degrades gracefully.
def apply_soft_stage1(probs_9class, p_rest_stage1, strength=1.0):
    probs=probs_9class.copy().astype(np.float64)
    p_rest=np.clip(np.asarray(p_rest_stage1),0.0,1.0)
    # boost Rest channel by stage-1 rest belief, damp active channels
    rest_factor   = (1.0 - strength) + strength * (p_rest)          # scales class 8
    active_factor = (1.0 - strength) + strength * (1.0 - p_rest)    # scales classes 0..7
    probs[:,8]   *= rest_factor
    probs[:,:8]  *= active_factor[:,None]
    probs /= (probs.sum(axis=1,keepdims=True)+1e-12)
    return probs

# ── Hard Stage-1 override (kept for USE_SOFT_STAGE1=False) ──────────────
def apply_stage1_override(probs_9class, is_active_mask):
    probs=probs_9class.copy()
    rest=~is_active_mask
    probs[rest,:]=0.0; probs[rest,8]=1.0
    act=is_active_mask; probs[act,8]=0.0
    probs[act]=probs[act]/(probs[act].sum(axis=1,keepdims=True)+1e-10)
    return probs

# ── ④ Temporal bout-consistency: collapse isolated 1s flickers ─────────
# Within each PID's time-ordered sequence, if a label differs from BOTH
# neighbors and both neighbors agree, replace it with the neighbor label.
# This encodes the physical prior that activities occur in contiguous bouts.
def apply_bout_consistency(labels, pids, times, passes=2):
    labels=np.asarray(labels).copy(); pids=np.asarray(pids); times=np.asarray(times)
    for _ in range(passes):
        for pid in np.unique(pids):
            idx=np.where(pids==pid)[0]
            if len(idx)<3: continue
            oidx=idx[np.argsort(times[idx],kind='mergesort')]
            seq=labels[oidx].copy()
            for i in range(1,len(seq)-1):
                if seq[i]!=seq[i-1] and seq[i-1]==seq[i+1]:
                    seq[i]=seq[i-1]
            labels[oidx]=seq
    return labels

print("Post-processing functions defined.")

In [ ]:
print("Tuning smoothing window + Stage1 mode on top blend candidates...")
top_cand=blend_df.head(15)
p_rest_oof = oof_binary_blend  # stage-1 rest probability for OOF

smooth_rows=[]
for _,row in top_cand.iterrows():
    w_l,w_c,w_e=row['w_lgbm'],row['w_cat'],row['w_et']
    raw=w_l*oof_lgbm+w_c*oof_cat+w_e*oof_et
    score_raw=balanced_accuracy_score(y_train,np.argmax(raw,1))

    best_score=score_raw; best_mode='raw'
    for window in [3,5,7]:
        # smoothed only
        sm=soft_prob_smooth(raw,groups,train_times,window=window)
        sc_sm=balanced_accuracy_score(y_train,np.argmax(sm,1))
        if sc_sm>best_score: best_score=sc_sm; best_mode=f'smooth_w{window}'

        # Stage-1 (soft or hard) + smooth
        if USE_SOFT_STAGE1:
            s1=apply_soft_stage1(raw, p_rest_oof, strength=1.0)
            tag='softs1'
        else:
            s1=apply_stage1_override(raw.copy(), oof_is_active)
            tag='hards1'
        s1s=soft_prob_smooth(s1,groups,train_times,window=window)
        sc_s1=balanced_accuracy_score(y_train,np.argmax(s1s,1))
        if sc_s1>best_score: best_score=sc_s1; best_mode=f'{tag}+smooth_w{window}'

    smooth_rows.append({'w_lgbm':w_l,'w_cat':w_c,'w_et':w_e,
                        'raw_oof_ba':score_raw,'best_oof_ba':best_score,'best_mode':best_mode})

smooth_df=pd.DataFrame(smooth_rows).sort_values('best_oof_ba',ascending=False)
best=smooth_df.iloc[0]
best_w_lgbm=float(best['w_lgbm']); best_w_cat=float(best['w_cat']); best_w_et=float(best['w_et'])
best_mode=best['best_mode']
print("\nTop blend+window candidates:")
print(smooth_df.head(10).to_string(index=False))
print(f"\nSelected: LGBM={best_w_lgbm:.2f} Cat={best_w_cat:.2f} ET={best_w_et:.2f} | mode={best_mode}")

In [ ]:
oof_blend  = best_w_lgbm*oof_lgbm  + best_w_cat*oof_cat  + best_w_et*oof_et
test_blend = best_w_lgbm*pred_lgbm + best_w_cat*pred_cat + best_w_et*pred_et

def parse_mode(s):
    """Returns (stage1_kind, window). stage1_kind in {None,'soft','hard'}"""
    if s=='raw': return None,5
    win=5
    if '_w' in s: win=int(s.split('_w')[-1])
    if s.startswith('softs1'): return 'soft',win
    if s.startswith('hards1'): return 'hard',win
    if s.startswith('smooth'): return None,win
    return None,5

s1_kind,best_window=parse_mode(best_mode)

# OOF final probs
if s1_kind=='soft':
    oof_p=apply_soft_stage1(oof_blend, oof_binary_blend, strength=1.0)
elif s1_kind=='hard':
    oof_p=apply_stage1_override(oof_blend.copy(), oof_is_active)
else:
    oof_p=oof_blend.copy()
if best_mode!='raw':
    oof_p=soft_prob_smooth(oof_p,groups,train_times,window=best_window)
final_oof_preds=np.argmax(oof_p,axis=1)

# TEST final probs
if s1_kind=='soft':
    test_p=apply_soft_stage1(test_blend, pred_binary_blend, strength=1.0)
elif s1_kind=='hard':
    test_p=apply_stage1_override(test_blend.copy(), test_is_active)
else:
    test_p=test_blend.copy()
if best_mode!='raw':
    test_p=soft_prob_smooth(test_p,test_label['pid'].values,test_times,window=best_window)
test_preds_final=np.argmax(test_p,axis=1)

# ④ Bout-consistency post-processing (validate on OOF before applying to test)
if USE_TEMPORAL_BOUT:
    ba_before=balanced_accuracy_score(y_train,final_oof_preds)
    oof_bout=apply_bout_consistency(final_oof_preds, groups, train_times, passes=2)
    ba_after=balanced_accuracy_score(y_train,oof_bout)
    print(f"Bout-consistency OOF: {ba_before:.5f} -> {ba_after:.5f}")
    if ba_after>=ba_before:   # only apply if it does NOT hurt OOF
        final_oof_preds=oof_bout
        test_preds_final=apply_bout_consistency(test_preds_final,
                                                test_label['pid'].values, test_times, passes=2)
        print("  Bout-consistency APPLIED to test.")
    else:
        print("  Bout-consistency skipped (would lower OOF).")

final_ba=balanced_accuracy_score(y_train,final_oof_preds)
print(f"\nFINAL OOF BA: {final_ba:.5f}")
print("(p5=0.91879, p6=0.91892)")

In [ ]:
class_names=['0:JumpJack','1:Jog','2:Squat','3:MtnClimb','4:PushUp',
             '5:Burpee','6:Lunge','7:JumpSquat','8:Rest']
p5_recalls=[0.967,0.943,0.882,0.891,0.940,0.942,0.886,0.902,0.917]
cm=confusion_matrix(y_train,final_oof_preds)
print("Per-class recall (p7 vs p5):")
for i,name in enumerate(class_names):
    r=cm[i,i]/cm[i].sum() if cm[i].sum()>0 else 0
    d=r-p5_recalls[i]; sign='+' if d>=0 else '-'
    print(f"  {name:14s}: {r:.3f}  {sign}{abs(d):.3f} vs p5")

In [ ]:
submission=pd.read_csv(os.path.join(DATA_DIR,'submission.csv'))
submission.loc[has_sensor_mask,'workout']=test_preds_final[has_sensor_mask]
submission.loc[pid71_mask,'workout']=8   # PID 71: no sensor data -> Rest
submission['workout']=submission['workout'].astype(int)

print("Submission preview:")
print(submission.head(10))
print("\nPrediction distribution:")
print(submission['workout'].value_counts().sort_index())

out_path=os.path.join(".",'submission_p9.csv')
submission.to_csv(out_path,index=False)
print(f"\nSaved -> {out_path}")
print("Done.")

In [1]:
import os
import pandas as pd

DATA_DIR = "/Users/dayana/git repo/machine-learning-class/lab6/knu-2026-machine-learning-final-assignment"

print("="*50)
print("FILES")
print("="*50)

for f in sorted(os.listdir(DATA_DIR)):
    print(f)

print("\n")
print("="*50)
print("CSV HEADERS")
print("="*50)

for f in sorted(os.listdir(DATA_DIR)):
    if f.endswith(".csv"):
        try:
            df = pd.read_csv(os.path.join(DATA_DIR, f), nrows=3)
            print(f"\n{f}")
            print(df.columns.tolist())
        except Exception as e:
            print(f"\n{f}")
            print("ERROR:", e)

FILES
submission.csv
test-accel.csv
test-gyro.csv
test-label.csv
train-accel.csv
train-gyro.csv
train-label.csv


CSV HEADERS

submission.csv
['id', 'workout']

test-accel.csv
['time', 'x', 'y', 'z', 'device', 'direction', 'pid']

test-gyro.csv
['time', 'x', 'y', 'z', 'device', 'direction', 'pid']

test-label.csv
['id', 'time', 'workout', 'pid']

train-accel.csv
['time', 'x', 'y', 'z', 'device', 'direction', 'pid']

train-gyro.csv
['time', 'x', 'y', 'z', 'device', 'direction', 'pid']

train-label.csv
['id', 'time', 'workout', 'pid']
